In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.data import random_split

import joblib
import json
from pathlib import Path
import copy

import sys
#sys.path.append("/work/home/acychkqsp6/zrm/MasterThesis/tools")
sys.path.append("../../tools")
import architec as acht
import evaluation as eva
import prepare as prep

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [2]:
import importlib
#importlib.reload(eva)
#importlib.reload(prep)


In [3]:
NumSpecies = 20
featgam = NumSpecies**2 - NumSpecies
task_num = 3 + 2 * NumSpecies # 0: num. of stable states, 1: stable-state entropy, 2: diversity, 3: abundance, 3+NumSpecies: switching species, 3+2*NumSpecies: fluctuation

# The folder storing datasets splited latter.
#folder_path = "/work/home/acychkqsp6/zrm/MasterThesis/Databases/Ver1.9/ver1.9.6_upto20"
#folder_path = "/work/home/acychkqsp6/zrm/MasterThesis/Databases/Ver1.9/ver1.9.6_upto20_fl"
folder_path = "../../Databases/Ver1.9/ver1.9.6_upto20_fl_v2"

for rep in range(1,4):
    #split_df = pd.read_csv(f"/work/home/acychkqsp6/zrm/MasterThesis/Principle_design_SynCom/Fig2/split_fl_v1/split_fl_{rep}.csv")
    #split_df = pd.read_csv(f"split_MD_v1/split_MD_{rep}.csv")
    split_df = pd.read_csv(f"../Fig2/split_ver1.9.6_upto20_fl_v2/split_{rep}.csv")

    #Trainset
    narray_tr, _ = prep.load_split_dataset2(
        split_df,
        folder_path,
        richness_list=[10],
        #richness_list=list(range(3,21)),
        subset="tr",
        sur_frac=False,
        MaxNum=NumSpecies,
    )
    #Testset
    narray_te, _ = prep.load_split_dataset2(
        split_df,
        folder_path,
        richness_list=[10],
        #richness_list=list(range(3,21)),
        subset="te",
        sur_frac=False,
        MaxNum=NumSpecies,
    )

    #-------------------------------------------------------
    Xg_tr, XA_tr, Xd_tr, y_tr, scaler_g, scaler_d, idx_tr = prep.prepare_train(
        narray_tr,
        NumSpecies,
        featgam,
        task_num,
        #sample_size=70000,
        random_state=42
    )

    Xg_te, XA_te, Xd_te, y_te, idx_te = prep.prepare_test(
        narray_te,
        NumSpecies,
        featgam,
        scaler_g,
        scaler_d,
        task_num,
        #sample_size=15000,
        random_state=42
    )

    train_dataset = acht.CommunityDataset(Xg_tr, XA_tr, Xd_tr, y_tr)
    test_dataset = acht.CommunityDataset(Xg_te, XA_te, Xd_te, y_te)

    n_val = int(len(train_dataset)*0.2)
    n_train = len(train_dataset)-n_val
    train_ds, val_ds = random_split(train_dataset, [n_train,n_val])

    train_loader = DataLoader(train_ds, batch_size=1024, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=1024, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)

    model = acht.MLP(NumSpecies, featgam, output_dim=1).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([1.0], device=device)) # positive class weight = 1
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    early_stop = acht.EarlyStopping(patience=8, min_delta=1e-4)
    history = {"loss":[], "val_loss":[]}

    best_state = None

    for epoch in range(140):
        model.train()
        train_loss = 0
        
        for Xg,XA,Xd,y in train_loader:

            Xg = Xg.to(device)
            XA = XA.to(device)
            Xd = Xd.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(Xg,XA,Xd)

            loss = criterion(logits,y)
            loss.backward()

            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        model.eval()

        val_loss = 0

        with torch.no_grad():

            for Xg,XA,Xd,y in val_loader:

                Xg = Xg.to(device)
                XA = XA.to(device)
                Xd = Xd.to(device)
                y = y.to(device)

                logits = model(Xg,XA,Xd)
                loss = criterion(logits,y)
                val_loss += loss.item()

        val_loss /= len(val_loader)

        history["loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        print(
            f"Epoch {epoch+1} "
            f"train={train_loss:.5f} "
            f"val={val_loss:.5f}"
        )

        if val_loss < early_stop.best_loss:
            best_state = copy.deepcopy(model.state_dict())

        if early_stop.step(val_loss):
            print("Early stopping")
            break

    model.load_state_dict(best_state)

    # Model saving
    #save_dir = Path(f"/work/home/acychkqsp6/zrm/MasterThesis/saved_models/fluctuation_15fold/sp3to20_Fl_transpt3_r{rep}")
    #save_dir = Path(f"../../saved_models/fluctuation_3fold_v0/sp10_Fl_transpt3_r{rep}")
    #save_dir = Path(f"../../saved_models/fluctuation_v2/sp10_Fl_transpt3_r{rep}")
    save_dir = Path(f"../../saved_models/fluctuation_3fold/sp10_Fl_MLP_r{rep}")

    save_dir.mkdir(parents=True, exist_ok=True)  # auto-create directories
    torch.save(model.state_dict(), save_dir / "model.pt")
    joblib.dump(scaler_g, save_dir / "scaler_g.pkl")
    joblib.dump(scaler_d, save_dir / "scaler_d.pkl")

    config = {
        "model_type": "MLP",
        "NumSpecies": NumSpecies,
        "featgam": featgam,
        "task_num": task_num,
        "hidden_dims": [512, 256, 128],
        "dropout": 0.1,
        "output_dim": 1
    }
    
    with open(save_dir / "config.json", "w") as f:
        json.dump(config, f)

    model.eval()
    preds = []

    with torch.no_grad():

        for Xg, XA, Xd, _ in test_loader:

            Xg = Xg.to(device)
            XA = XA.to(device)
            Xd = Xd.to(device)

            logits = model(Xg,XA,Xd)
            prob = torch.sigmoid(logits)
            preds.append(prob.cpu().numpy())

    y_prob = np.concatenate(preds).ravel()

    threshold=0.8
    y_pred = (y_prob >= threshold).astype(int)

    metrics = eva.eval_binary(y_te, y_prob, threshold=threshold)
    df_metrics = pd.DataFrame([metrics])
    print(df_metrics.to_string())
    

tr shape = (113980, 465)
te shape = (28494, 465)
Epoch 1 train=0.59017 val=0.53252
Epoch 2 train=0.51086 val=0.50037
Epoch 3 train=0.46712 val=0.47321
Epoch 4 train=0.42664 val=0.48702
Epoch 5 train=0.38751 val=0.43100
Epoch 6 train=0.36039 val=0.43451
Epoch 7 train=0.33272 val=0.40931
Epoch 8 train=0.31058 val=0.41429
Epoch 9 train=0.29484 val=0.40265
Epoch 10 train=0.27197 val=0.41440
Epoch 11 train=0.25586 val=0.40995
Epoch 12 train=0.24579 val=0.41273
Epoch 13 train=0.24128 val=0.44960
Epoch 14 train=0.22863 val=0.41987
Epoch 15 train=0.21724 val=0.43798
Epoch 16 train=0.21028 val=0.41935
Epoch 17 train=0.20436 val=0.44848
Early stopping
   Accuracy  Balanced_Acc        F1       MCC       AUC     AUPRC  AUPRC_rel     Brier  pos_rate
0  0.805678      0.734416  0.641641  0.565679  0.896823  0.837451   1.400877  0.117212   0.34881
tr shape = (113980, 465)
te shape = (28494, 465)
Epoch 1 train=0.59558 val=0.53169
Epoch 2 train=0.50918 val=0.51595
Epoch 3 train=0.46602 val=0.48239
Epoch